# Looking at data

In [ ]:
import os

import hydra
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import seaborn as sns
import torch
from hydra.utils import instantiate
from IPython.display import HTML
from tqdm import tqdm

from experanto.dataloaders import get_multisession_concat_dataloader
from experanto.utils import handle_responses_field
from experanto.configs import BENCHMARKING_CONFIG as cfg

%matplotlib inline
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

# Load configs and data

In [2]:
# Handle the responses field: we specify a specific filtering in the config, but experanto needs a clean "responses" field.
cfg = handle_responses_field(cfg)

# get train/validation loader with 100 samples ~=3s of context

In [3]:
paths = ["/mnt/data1/enigma/goliath_10_20_sandbox/37_3843837605846_0_V3A_V4/",]
cfg.dataset.modality_config.screen.valid_condition = {"tier": "train"}
cfg.dataset.modality_config.screen.include_blanks = True
cfg.dataset.modality_config.screen.sample_stride = 1
train_dl = get_multisession_concat_dataloader(paths, cfg)
cfg.dataset.modality_config.screen.sample_stride = cfg.dataset.modality_config.screen.chunk_size
# override blanks for validation
cfg.dataset.modality_config.screen.include_blanks = False
cfg.dataset.modality_config.screen.valid_condition = {"tier": "validation"}
val_dl = get_multisession_concat_dataloader(paths, cfg)

/usr/local/lib/python3.12/dist-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Dataset 0: 37_3843837605846_0_V3A_V4, length = 168584
Sessions: ['37_3843837605846_0_V3A_V4']
Batches per session: {'37_3843837605846_0_V3A_V4': 21073}
Total batches: 21073
Created FastSessionDataLoader with 1 sessions and 21073 total batches
Dataset 0: 37_3843837605846_0_V3A_V4, length = 177
Sessions: ['37_3843837605846_0_V3A_V4']
Batches per session: {'37_3843837605846_0_V3A_V4': 22}
Total batches: 22
Created FastSessionDataLoader with 1 sessions and 22 total batches


# get test loader with 15 samples =0.5s of context (test videos with fixation are only 0.5 seconds long)

In [4]:
cs = 15
cfg.dataset.modality_config.screen.chunk_size = cs
cfg.dataset.modality_config.eye_tracker.chunk_size = cs
cfg.dataset.modality_config.responses.chunk_size = cs
cfg.dataset.modality_config.blinks.chunk_size = cs
cfg.dataset.modality_config.screen.include_blanks = True
cfg.dataset.modality_config.screen.sample_stride = cs
cfg.dataset.modality_config.screen.valid_condition = {"tier": "test"}
test_dl = get_multisession_concat_dataloader(paths, cfg)

Dataset 0: 37_3843837605846_0_V3A_V4, length = 435
Sessions: ['37_3843837605846_0_V3A_V4']
Batches per session: {'37_3843837605846_0_V3A_V4': 54}
Total batches: 54
Created FastSessionDataLoader with 1 sessions and 54 total batches


# visualize training batches in free viewing

In [6]:
data_key, batch = next(iter(train_dl))
# run this cell again to get more examples